# Inspect `slst` SVG generation

This notebook exercises only the sampling and SVG-drawing stages. It does not rasterize, calculate OBB labels, or write artifacts to disk.

In [ ]:
from pathlib import Path
import sys

import numpy as np
from IPython.display import SVG, display

PROJECT_ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / '06_configs' / 'ontology.yaml').exists()),
    Path.cwd(),
)
SCRIPTS_ROOT = PROJECT_ROOT / '01_scripts'
if str(SCRIPTS_ROOT) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_ROOT))

from generation.core.models import GenerationConfig
from generation.core.sampling import load_sampling_config
from generation.registry import GENERATOR_REGISTRY


In [ ]:
ontology_path = PROJECT_ROOT / '06_configs' / 'ontology.yaml'
sampling_path = PROJECT_ROOT / '06_configs' / 'symbol_sampling.yaml'

sampling_config = load_sampling_config(
    ontology_path=ontology_path,
    sampling_path=sampling_path,
)

class_key = ('primitive', 'slst')
spec = sampling_config.resolve(*class_key)
generator = GENERATOR_REGISTRY[class_key]
print(f'Generator: {generator.__name__}')
print(f'Class ID: {spec.class_id}')
print(f'Supported registry keys: {list(GENERATOR_REGISTRY)}')

## One explicit case

Change the override mapping below to inspect another concrete sample.

In [ ]:
seed = 1234
overrides = {
    'shape': 'oval',
    'aspect_ratio': 1.15,
}
generation_config = GenerationConfig(
    canvas_width_px=150,
    canvas_height_px=150,
    target_visible_px=90,
    rotation_deg=20,
    stroke_width_normalized=4.0,
)

sample = sampling_config.sample(
    *class_key,
    np.random.default_rng(seed),
    seed=seed,
    overrides=overrides,
)
generated = generator(spec, sample, generation_config)

display(SVG(data=generated.svg))
print('Sampled parameters:', sample.parameters)
print('Derived parameters:', sample.derived)
print('Metadata:', generated.metadata)

## Compare configured shape branches

Both branches preserve their explicit aspect-ratio override.

In [ ]:
cases = [
    ('oval', 1.15, 2001),
    ('circle', 1.04, 2002),
]

for shape, aspect_ratio, seed in cases:
    sample = sampling_config.sample(
        *class_key,
        np.random.default_rng(seed),
        seed=seed,
        overrides={'shape': shape, 'aspect_ratio': aspect_ratio},
    )
    generated = generator(spec, sample, GenerationConfig(
        canvas_width_px=150,
        canvas_height_px=150,
        target_visible_px=90,
        rotation_deg=0,
    ))
    print(f'{shape=}, {aspect_ratio=}, sampled={sample.as_dict()}')
    display(SVG(data=generated.svg))

## Optional: save one SVG for external inspection

Run this cell only when a durable SVG file is useful. This writes SVG text directly; it does not rasterize or label it.

In [ ]:
# output_path = PROJECT_ROOT / '02_notebooks' / 'generation' / 'outputs' / 'slst_inspection.svg'
# output_path.parent.mkdir(parents=True, exist_ok=True)
# output_path.write_text(generated.svg, encoding='utf-8')
# print(output_path)